##### 3.1.3.3 person_income

**Входы**

- Столбец person_income после очистки

**Выходы**

- Таблицы, графики и наблюдения по признаку

**Что делаем**

Посмотрим, какие значения встречаются чаще и что выделяется на их фоне

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


##### Годовой доход — person_income

Сначала посмотрим статистику и распределение дохода в обычном масштабе

In [ ]:
import numpy as np
from matplotlib.ticker import FuncFormatter

income = data['person_income']
income_q1, income_q3 = income.quantile([0.25, 0.75])
income_iqr = income_q3 - income_q1
income_lower = income_q1 - 1.5 * income_iqr
income_upper = income_q3 + 1.5 * income_iqr
income_outliers = (income < income_lower) | (income > income_upper)

income_stats = pd.concat([
    income.describe(),
    pd.Series({
        'Коэффициент асимметрии': income.skew(),
        'Верхняя граница по правилу 1,5 IQR': income_upper,
        'Записей за границами 1,5 IQR': int(income_outliers.sum()),
        'Доля записей за границами, %': 100 * income_outliers.mean(),
    }),
]).to_frame('Значение')
display(income_stats)

money_format = FuncFormatter(lambda value, _: f'{value:,.0f}'.replace(',', ' '))
fig, income_linear_axes = plt.subplots(2, 1, figsize=(13, 7),
                                     gridspec_kw={'height_ratios': [3, 1]}, constrained_layout=True)
sns.histplot(x=income, bins=50, color='#3977a8', ax=income_linear_axes[0])
income_linear_axes[0].set(title='Распределение годового дохода',
                          xlabel='Годовой доход', ylabel='Количество записей')
income_linear_axes[0].xaxis.set_major_formatter(money_format)
income_linear_axes[0].tick_params(axis='x', labelrotation=20)
sns.boxplot(x=income, color='#91bad7', ax=income_linear_axes[1])
income_linear_axes[1].set(title='Доход — ящик с усами', xlabel='Годовой доход')
income_linear_axes[1].xaxis.set_major_formatter(money_format)
plt.show()


##### Вывод по обычной шкале

Длинный правый хвост растягивает ось дохода: основная часть записей сжата у левого края, и её распределение плохо различимо. Для подробного просмотра построим гистограмму в логарифмическом масштабе. Меняем только шкалу графика, значения признака сохраняем

In [ ]:
fig, income_log_ax = plt.subplots(figsize=(13, 4.5), constrained_layout=True)
log_bins = np.geomspace(income.min(), income.max(), 40)
sns.histplot(x=income, bins=log_bins, color='#3977a8', ax=income_log_ax)
income_log_ax.set_xscale('log')
income_log_ax.axvline(income.median(), color='#b34c32', linestyle='--',
                      label=f'Медиана: {income.median():,.0f}'.replace(',', ' '))
income_log_ax.set(title='Доход — логарифмическая шкала',
                   xlabel='Годовой доход, логарифмическая шкала', ylabel='Количество записей')
income_log_ax.xaxis.set_major_formatter(money_format)
income_log_ax.legend()
plt.show()


**Выводы и решения по person_income**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f16', 'У дохода длинный правый хвост — медиана 55 000, встречаются редкие доходы свыше миллиона', 'Проверить, отличаются ли редкие высокие доходы по доле дефолтов, прежде чем решать, как их обрабатывать.\nЗаёмщики с высокими доходами могут существенно отличаться по кредитному поведению — эту гипотезу предстоит проверить'),
])
display(findings_view(conclusions, only_new=True))